In [1]:
# ============================================================
# VLC p(y|x) multimodality batch checker (Jupyter-safe, 1 cell)
# Scans: /workspace/2026/dataset_fullsquare_organized/**/IQ_data/
# Loads: sent_data_tuple.npy (X), received_data_tuple_sync-phase.npy (Y)
# Outputs:
#   /workspace/2026/multimodality_results/<dataset_tag>/multimodality_summary.csv
#   /workspace/2026/multimodality_results/global_multimodality_inventory.csv
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

# ----------------------------
# 0) CONFIG (edit only here)
# ----------------------------
ROOT = Path("/workspace/2026/dataset_fullsquare_organized")
OUT_ROOT = Path("/workspace/2026/multimodality_results")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CFG = {
    # filenames inside IQ_data
    "x_filename": "sent_data_tuple.npy",
    "y_filename": "received_data_tuple_sync-phase.npy",

    # grouping for continuous X (full-square)
    "mode": "grid",            # "grid" (recommended) or "knn"
    "min_group_size": 2000,

    # grid params
    "grid_bins_I": 15,
    "grid_bins_Q": 15,

    # knn params (if mode="knn")
    "knn_num_anchors": 60,
    "knn_k_neighbors": 4000,
    "knn_seed": 123,

    # outlier filtering on Y within each group
    "center_method": "median",     # "median" or "mean"
    "outlier_clip_mad": 12.0,      # None disables
    "outlier_clip_tail_frac": 0.0, # e.g., 0.001

    # GMM/BIC
    "gmm_k_max": 6,
    "gmm_covariance_type": "full",
    "gmm_n_init": 2,
    "gmm_max_iter": 250,
    "gmm_reg_covar": 1e-6,
    "gmm_seed": 123,

    # thresholds for evidence from BIC
    "bic_strong_threshold": 10.0,
    "bic_very_strong_threshold": 30.0,

    # BOOTSTRAP: set to 0 for fast triage
    "bootstrap_B": 0,      # 0 disables bootstrap
    "bootstrap_n": 2500,
    "bootstrap_seed": 123,

    # Optional diptest (only if installed)
    "use_diptest_if_available": True,
    "diptest_alpha": 0.05,

    # plots (batch default off)
    "save_plots": False,
    "max_plots": 6,
}

# ----------------------------
# 1) Robust MAD (no SciPy dependency)
# ----------------------------
def median_abs_deviation(x, scale="normal"):
    x = np.asarray(x)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if scale == "normal":
        return 1.4826 * mad
    return mad

# ----------------------------
# 2) Optional diptest import
# ----------------------------
diptest_available = False
_diptest = None
if CFG["use_diptest_if_available"]:
    try:
        from diptest import diptest as _diptest
        diptest_available = True
        print("[info] diptest available.")
    except Exception:
        diptest_available = False
        print("[warn] diptest not available (pip install diptest). Using only GMM/BIC (+ optional bootstrap).")

def dip_pvalue(x1d):
    if not diptest_available:
        return np.nan
    x1d = np.asarray(x1d).ravel()
    if x1d.size < 20:
        return np.nan
    _, p = _diptest(x1d)
    return float(p)

# ----------------------------
# 3) Shape & sanity helpers
# ----------------------------
def ensure_N2(A, name="A"):
    A = np.asarray(A)
    if A.ndim != 2:
        raise ValueError(f"{name} must be 2D. Got {A.ndim}D shape={A.shape}")
    if A.shape[1] == 2:
        return A
    if A.shape[0] == 2:
        return A.T
    raise ValueError(f"{name} not I/Q-like. shape={A.shape} (expected (N,2) or (2,N))")

def robust_center(A, method="median"):
    return np.median(A, axis=0) if method == "median" else np.mean(A, axis=0)

def clip_outliers(Yg, center_method="median", clip_mad=None, tail_frac=0.0):
    if Yg.shape[0] < 10:
        return Yg
    c = robust_center(Yg, center_method)
    Z = Yg - c

    # MAD radial clipping
    if clip_mad is not None:
        r = np.sqrt(np.sum(Z**2, axis=1))
        s = median_abs_deviation(r, scale="normal")
        if s > 0:
            thr = np.median(r) + clip_mad * s
            keep = r <= thr
            Yg = Yg[keep]
            if Yg.shape[0] < 10:
                return Yg
            c = robust_center(Yg, center_method)
            Z = Yg - c

    # Tail drop
    if tail_frac and tail_frac > 0:
        r = np.sqrt(np.sum(Z**2, axis=1))
        thr = np.quantile(r, 1.0 - tail_frac)
        keep = r <= thr
        Yg = Yg[keep]

    return Yg

def projections_1d(Yg, center_method="median"):
    c = robust_center(Yg, center_method)
    Z = Yg - c
    yi = Z[:, 0]
    yq = Z[:, 1]
    r = np.sqrt(yi**2 + yq**2)
    if Z.shape[0] >= 3:
        p1 = PCA(n_components=1).fit_transform(Z).ravel()
    else:
        p1 = r.copy()
    return {"I": yi, "Q": yq, "R": r, "PCA1": p1}

# ----------------------------
# 4) GMM+BIC core
# ----------------------------
def fit_gmm_bic(Yg, cfg, seed=0):
    k_max = int(cfg["gmm_k_max"])
    bics = []
    models = []
    for k in range(1, k_max + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type=cfg["gmm_covariance_type"],
            n_init=cfg["gmm_n_init"],
            max_iter=cfg["gmm_max_iter"],
            reg_covar=cfg["gmm_reg_covar"],
            random_state=seed,
        )
        gmm.fit(Yg)
        bics.append(float(gmm.bic(Yg)))
        models.append(gmm)

    bics = np.array(bics, dtype=float)
    k_best = int(np.argmin(bics) + 1)

    bic1 = float(bics[0])
    bic_min_kge2 = float(np.min(bics[1:])) if k_max >= 2 else float("inf")
    delta_bic = bic1 - bic_min_kge2 if k_max >= 2 else 0.0

    return {
        "k_best": k_best,
        "bics": bics,
        "bic1": bic1,
        "delta_bic": float(delta_bic),
        "model_best": models[k_best - 1],
    }

# ----------------------------
# 5) Bootstrap (optional)
# ----------------------------
def bootstrap_group(Yg, cfg, seed_base=123):
    B = int(cfg["bootstrap_B"])
    if B <= 0:
        return {
            "boot_frac_k_ge2": np.nan,
            "boot_delta_bic_median": np.nan,
            "boot_delta_bic_p10": np.nan,
            "boot_dip_p_median": np.nan,
            "boot_dip_reject_frac": np.nan,
        }

    rng = np.random.default_rng(seed_base)
    n_boot = int(cfg["bootstrap_n"])

    m = Yg.shape[0]
    if m < cfg["min_group_size"]:
        return {
            "boot_frac_k_ge2": np.nan,
            "boot_delta_bic_median": np.nan,
            "boot_delta_bic_p10": np.nan,
            "boot_dip_p_median": np.nan,
            "boot_dip_reject_frac": np.nan,
        }

    n_boot = min(n_boot, m)

    k_bests = []
    delta_bics = []
    dip_mins = []

    for b in range(B):
        idx = rng.integers(0, m, size=n_boot)
        Ys = Yg[idx]

        g = fit_gmm_bic(Ys, cfg, seed=int(seed_base + b))
        k_bests.append(g["k_best"])
        delta_bics.append(g["delta_bic"])

        if diptest_available:
            proj = projections_1d(Ys, cfg["center_method"])
            pvals = [dip_pvalue(proj[name]) for name in proj]
            pvals = [p for p in pvals if np.isfinite(p)]
            dip_mins.append(np.min(pvals) if len(pvals) else np.nan)

    k_bests = np.array(k_bests)
    delta_bics = np.array(delta_bics, dtype=float)

    out = {
        "boot_frac_k_ge2": float(np.mean(k_bests >= 2)),
        "boot_delta_bic_median": float(np.nanmedian(delta_bics)),
        "boot_delta_bic_p10": float(np.nanpercentile(delta_bics, 10)),
    }

    if diptest_available:
        dip_mins = np.array(dip_mins, dtype=float)
        out.update({
            "boot_dip_p_median": float(np.nanmedian(dip_mins)),
            "boot_dip_reject_frac": float(np.mean(dip_mins < cfg["diptest_alpha"])),
        })
    else:
        out.update({
            "boot_dip_p_median": np.nan,
            "boot_dip_reject_frac": np.nan,
        })

    return out

# ----------------------------
# 6) Grouping (grid / knn)
# ----------------------------
def make_groups_grid(X, bins_I=15, bins_Q=15):
    xi = X[:, 0]
    xq = X[:, 1]
    ei = np.linspace(xi.min(), xi.max(), bins_I + 1)
    eq = np.linspace(xq.min(), xq.max(), bins_Q + 1)
    bi = np.clip(np.digitize(xi, ei) - 1, 0, bins_I - 1)
    bq = np.clip(np.digitize(xq, eq) - 1, 0, bins_Q - 1)

    centers = {}
    for iI in range(bins_I):
        ci = 0.5 * (ei[iI] + ei[iI + 1])
        for iQ in range(bins_Q):
            cq = 0.5 * (eq[iQ] + eq[iQ + 1])
            centers[f"binI{iI:02d}_binQ{iQ:02d}"] = (float(ci), float(cq))

    groups = {}
    for i in range(X.shape[0]):
        k = f"binI{bi[i]:02d}_binQ{bq[i]:02d}"
        groups.setdefault(k, []).append(i)

    groups = {k: np.asarray(v, dtype=int) for k, v in groups.items()}
    return groups, centers

def make_groups_knn(X, num_anchors=60, k_neighbors=4000, seed=0):
    rng = np.random.default_rng(seed)
    N = X.shape[0]
    num_anchors = min(num_anchors, N)

    anchor_idx = rng.choice(N, size=num_anchors, replace=False)
    anchors = X[anchor_idx]

    k_neighbors = min(k_neighbors, N)
    nn = NearestNeighbors(n_neighbors=k_neighbors, algorithm="auto")
    nn.fit(X)
    _, neigh = nn.kneighbors(anchors, return_distance=True)

    groups = {}
    centers = {}
    for a, idxs in enumerate(neigh):
        k = f"anchor{a:03d}"
        groups[k] = idxs.astype(int)
        centers[k] = (float(anchors[a, 0]), float(anchors[a, 1]))
    return groups, centers

# ----------------------------
# 7) Dataset discovery
# ----------------------------
#iq_dirs = sorted([p for p in ROOT.rglob("IQ_data") if p.is_dir()])
# ============================================================
# Escolha manual de 1 dataset
# ============================================================

iq_dirs = [
    Path("/workspace/2026/dataset_fullsquare_organized/"
         "dist_0.8m/curr_100mA/"
         "full_square_0.8m_100mA_001_20260211_153055/"
         "full_square_0.8m_100mA_001_20260211_153055/IQ_data")
]

print(f"[info] Running only 1 dataset: {iq_dirs[0]}")

print(f"[info] Found {len(iq_dirs)} IQ_data folders under: {ROOT}")

def dataset_tag_from_iq_dir(iq_dir: Path) -> str:
    parts = iq_dir.parts
    dist = next((p for p in parts if p.startswith("dist_")), "dist_unknown")
    curr = next((p for p in parts if p.startswith("curr_")), "curr_unknown")
    run = iq_dir.parent.name
    return f"{dist}__{curr}__{run}"

# ----------------------------
# 8) Main loop
# ----------------------------
global_rows = []

for di, iq_dir in enumerate(iq_dirs, start=1):
    tag = dataset_tag_from_iq_dir(iq_dir)
    out_dir = OUT_ROOT / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    x_path = iq_dir / CFG["x_filename"]
    y_path = iq_dir / CFG["y_filename"]

    if not x_path.exists() or not y_path.exists():
        print(f"[skip] Missing npy in: {iq_dir}")
        continue

    print(f"\n[{di}/{len(iq_dirs)}] {tag}")

    # Load X,Y
    try:
        X = ensure_N2(np.load(x_path), "X (sent)")
        Y = ensure_N2(np.load(y_path), "Y (received)")
    except Exception as e:
        print(f"[skip] load/shape error: {e}")
        continue

    if X.shape != Y.shape:
        print(f"[skip] shape mismatch X{X.shape} != Y{Y.shape}")
        continue
    if (not np.isfinite(X).all()) or (not np.isfinite(Y).all()):
        print("[skip] NaN/Inf detected")
        continue

    # Grouping
    if CFG["mode"].lower() == "grid":
        groups, centers = make_groups_grid(X, CFG["grid_bins_I"], CFG["grid_bins_Q"])
    else:
        groups, centers = make_groups_knn(X, CFG["knn_num_anchors"], CFG["knn_k_neighbors"], CFG["knn_seed"])

    group_items = sorted(groups.items(), key=lambda kv: kv[1].size, reverse=True)

    plot_keys = set([k for k, _ in group_items[:CFG["max_plots"]]]) if CFG["save_plots"] else set()

    rows = []
    n_groups_used = 0
    n_groups_multimodal = 0

    for gi, (gkey, idx) in enumerate(group_items, start=1):
        m_raw = idx.size
        if m_raw < CFG["min_group_size"]:
            continue

        Yg = Y[idx]
        Yg = clip_outliers(
            Yg,
            center_method=CFG["center_method"],
            clip_mad=CFG["outlier_clip_mad"],
            tail_frac=CFG["outlier_clip_tail_frac"],
        )

        m_used = Yg.shape[0]
        if m_used < CFG["min_group_size"]:
            continue

        n_groups_used += 1

        # Dip (optional)
        dip_min_p = np.nan
        dip_reject = np.nan
        if diptest_available:
            proj = projections_1d(Yg, CFG["center_method"])
            pvals = [dip_pvalue(proj[name]) for name in proj]
            pvals = [p for p in pvals if np.isfinite(p)]
            dip_min_p = float(np.min(pvals)) if len(pvals) else np.nan
            dip_reject = bool(dip_min_p < CFG["diptest_alpha"]) if np.isfinite(dip_min_p) else np.nan

        # GMM/BIC
        gmm = fit_gmm_bic(Yg, CFG, seed=CFG["gmm_seed"] + gi + di * 10000)
        delta_bic = gmm["delta_bic"]

        bic_flag = (
            "very_strong" if delta_bic >= CFG["bic_very_strong_threshold"]
            else "strong" if delta_bic >= CFG["bic_strong_threshold"]
            else "weak_or_none"
        )

        # Bootstrap (optional)
        boot = bootstrap_group(Yg, CFG, seed_base=CFG["bootstrap_seed"] + gi + di * 10000)

        # Multimodal decision:
        # - If bootstrap disabled: rely on (k_best>=2 and delta_bic>=10)
        # - If bootstrap enabled: add stability requirements
        if int(CFG["bootstrap_B"]) <= 0:
            multimodal = (gmm["k_best"] >= 2) and (delta_bic >= CFG["bic_strong_threshold"])
        else:
            multimodal = (
                (gmm["k_best"] >= 2) and
                (boot["boot_frac_k_ge2"] >= 0.7) and
                (boot["boot_delta_bic_median"] >= CFG["bic_strong_threshold"])
            )

        if multimodal:
            n_groups_multimodal += 1

        cx, cq = centers.get(gkey, (np.nan, np.nan))

        rows.append({
            "dataset_tag": tag,
            "iq_dir": str(iq_dir),
            "group_key": gkey,
            "group_size_raw": int(m_raw),
            "group_size_used": int(m_used),
            "x_center_I": float(cx),
            "x_center_Q": float(cq),

            "dip_min_p": dip_min_p,
            "dip_reject_alpha": dip_reject,

            "gmm_k_best": int(gmm["k_best"]),
            "gmm_delta_bic": float(delta_bic),
            "gmm_bic_flag": bic_flag,

            "boot_frac_k_ge2": boot["boot_frac_k_ge2"],
            "boot_delta_bic_median": boot["boot_delta_bic_median"],
            "boot_delta_bic_p10": boot["boot_delta_bic_p10"],
            "boot_dip_reject_frac": boot["boot_dip_reject_frac"],

            "multimodal_decision": bool(multimodal),
        })

        # Plots (optional)
        if CFG["save_plots"] and gkey in plot_keys:
            fig, ax = plt.subplots(figsize=(6, 6))
            ax.scatter(Yg[:, 0], Yg[:, 1], s=2, alpha=0.35)
            title = f"{tag} | {gkey} | n={m_used} | k_best={gmm['k_best']} | ΔBIC={delta_bic:.1f}"
            if diptest_available and np.isfinite(dip_min_p):
                title += f" | dip={dip_min_p:.3g}"
            ax.set_title(title)
            ax.set_xlabel("y_I")
            ax.set_ylabel("y_Q")
            ax.grid(True, alpha=0.3)
            fig.tight_layout()
            fig.savefig(out_dir / f"scatter_{gkey}.png", dpi=180)
            plt.close(fig)

    # Save per-dataset CSV
    df = pd.DataFrame(rows)
    per_csv = out_dir / "multimodality_summary.csv"
    df.to_csv(per_csv, index=False)

    frac_multimodal = (n_groups_multimodal / n_groups_used) if n_groups_used > 0 else np.nan

    # dataset-level verdict: "serves multimodality argument for cVAE?"
    # conservative: require at least 10% of used groups flagged multimodal
    dataset_multimodal = bool(frac_multimodal >= 0.10) if np.isfinite(frac_multimodal) else False

    global_rows.append({
        "dataset_tag": tag,
        "iq_dir": str(iq_dir),
        "N_samples": int(X.shape[0]),
        "groups_used": int(n_groups_used),
        "groups_multimodal": int(n_groups_multimodal),
        "frac_groups_multimodal": float(frac_multimodal) if np.isfinite(frac_multimodal) else np.nan,
        "dataset_multimodal_decision": dataset_multimodal,
        "bootstrap_B": int(CFG["bootstrap_B"]),
        "grid_bins_I": int(CFG["grid_bins_I"]) if CFG["mode"].lower()=="grid" else np.nan,
        "grid_bins_Q": int(CFG["grid_bins_Q"]) if CFG["mode"].lower()=="grid" else np.nan,
    })

    print(f"  -> groups_used={n_groups_used}, multimodal_groups={n_groups_multimodal}, frac={frac_multimodal:.3f}")
    print(f"  -> saved: {per_csv}")

# Save global inventory
global_df = pd.DataFrame(global_rows)
global_df = global_df.sort_values(
    by=["dataset_multimodal_decision", "frac_groups_multimodal", "groups_used"],
    ascending=[False, False, False],
).reset_index(drop=True)

global_csv = OUT_ROOT / "global_multimodality_inventory.csv"
global_df.to_csv(global_csv, index=False)

print(f"\n[done] Saved GLOBAL inventory: {global_csv}")
global_df.head(30)


[warn] diptest not available (pip install diptest). Using only GMM/BIC (+ optional bootstrap).
[info] Running only 1 dataset: /workspace/2026/dataset_fullsquare_organized/dist_0.8m/curr_100mA/full_square_0.8m_100mA_001_20260211_153055/full_square_0.8m_100mA_001_20260211_153055/IQ_data
[info] Found 1 IQ_data folders under: /workspace/2026/dataset_fullsquare_organized

[1/1] dist_0.8m__curr_100mA__full_square_0.8m_100mA_001_20260211_153055
  -> groups_used=225, multimodal_groups=0, frac=0.000
  -> saved: /workspace/2026/multimodality_results/dist_0.8m__curr_100mA__full_square_0.8m_100mA_001_20260211_153055/multimodality_summary.csv

[done] Saved GLOBAL inventory: /workspace/2026/multimodality_results/global_multimodality_inventory.csv


,dataset_tag,iq_dir,N_samples,groups_used,groups_multimodal,frac_groups_multimodal,dataset_multimodal_decision,bootstrap_B,grid_bins_I,grid_bins_Q
0,dist_0.8m__curr_100mA__full_square_0.8m_100mA_...,/workspace/2026/dataset_fullsquare_organized/d...,900000,225,0,0.0,False,0,15,15
